# Step 2: Data Collection and Understanding

This notebook documents the source dataset, profiles its structure and quality, and creates a data dictionary for the eCommerce purchase-propensity modeling project.

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

# Identify the current notebook folder and locate the project dataset.
current_folder = Path.cwd().resolve()

possible_data_paths = [
    current_folder / "data" / "raw" / "2020-Apr.csv",
    current_folder.parent / "data" / "raw" / "2020-Apr.csv"
]

DATA_PATH = next((path for path in possible_data_paths if path.exists()), None)

if DATA_PATH is None:
    searched_paths = "\n".join(str(path) for path in possible_data_paths)
    raise FileNotFoundError(
        "Dataset file was not found. Confirm that it is stored in:\n"
        "project_folder/data/raw/2020-Apr.csv\n\n"
        f"Paths checked:\n{searched_paths}"
    )

PROCESSED_DATA_FOLDER = DATA_PATH.parent.parent / "processed"
PROCESSED_DATA_FOLDER.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 250_000

# Keep approximately 0.25% of sessions for lightweight profiling.
# The hash rule makes the sample reproducible.
HASH_MODULUS = 400
HASH_BUCKET = 0

print(f"Working directory: {current_folder}")
print(f"Dataset found: {DATA_PATH}")
print(f"Processed-data folder: {PROCESSED_DATA_FOLDER}")

Working directory: C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\notebooks
Dataset found: C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\data\raw\2020-Apr.csv
Processed-data folder: C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\data\processed


The dataset path configuration was validated successfully. The notebook correctly identified the raw dataset stored in `data/raw/2020-Apr.csv` and confirmed that `data/processed/` is available for storing generated documentation outputs. This structure supports a reproducible project workflow by separating raw source data from derived datasets and analysis artifacts.

In [2]:
required_columns = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session"
]

column_dtypes = {
    "event_time": "string",
    "event_type": "string",
    "product_id": "string",
    "category_id": "string",
    "category_code": "string",
    "brand": "string",
    "price": "float64",
    "user_id": "string",
    "user_session": "string"
}

data_dictionary_metadata = {
    "event_time": {
        "description": "UTC timestamp when the customer event occurred.",
        "semantic_type": "Datetime / temporal feature",
        "format_or_unit": "UTC date and time",
        "modeling_role": "Potential feature for deriving time-based variables"
    },
    "event_type": {
        "description": "Type of customer interaction with a product.",
        "semantic_type": "Categorical behavioral event",
        "format_or_unit": "Observed event labels",
        "modeling_role": "Potential feature; purchase events will later define the target"
    },
    "product_id": {
        "description": "Unique identifier assigned to a product.",
        "semantic_type": "Identifier",
        "format_or_unit": "Product ID",
        "modeling_role": "Potential feature after aggregation or encoding"
    },
    "category_id": {
        "description": "Unique identifier assigned to a product category.",
        "semantic_type": "Identifier",
        "format_or_unit": "Category ID",
        "modeling_role": "Potential feature after aggregation or encoding"
    },
    "category_code": {
        "description": "Hierarchical text label describing the product category.",
        "semantic_type": "Categorical text feature",
        "format_or_unit": "Dot-separated category hierarchy, when available",
        "modeling_role": "Potential feature after cleaning or category-level aggregation"
    },
    "brand": {
        "description": "Product brand name, recorded in lowercase when available.",
        "semantic_type": "Categorical feature",
        "format_or_unit": "Brand name",
        "modeling_role": "Potential feature after missing-value treatment and encoding"
    },
    "price": {
        "description": "Recorded price of the product associated with the event.",
        "semantic_type": "Continuous numeric feature",
        "format_or_unit": "Price value; currency is not confirmed in the source data",
        "modeling_role": "Potential numeric feature and source for derived aggregates"
    },
    "user_id": {
        "description": "Permanent identifier assigned to a user.",
        "semantic_type": "Identifier",
        "format_or_unit": "User ID",
        "modeling_role": "Not intended for direct model use; may support behavioral aggregation"
    },
    "user_session": {
        "description": "Identifier representing one user browsing session.",
        "semantic_type": "Session identifier",
        "format_or_unit": "UUID-like session ID",
        "modeling_role": "Primary grouping field and future unit of analysis"
    }
}

In [3]:
total_rows = 0
missing_counts = Counter()
blank_string_counts = Counter()
event_type_counts = Counter()

invalid_datetime_count = 0
earliest_event_time = None
latest_event_time = None

price_missing_count = 0
price_non_positive_count = 0
price_valid_count = 0
price_min = np.inf
price_max = -np.inf

sampled_rows = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=required_columns,
        dtype=column_dtypes,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows += len(chunk)

    # Exact null counts across the full dataset.
    for column in required_columns:
        missing_counts[column] += int(chunk[column].isna().sum())

    # Blank-value counts are checked separately from null values.
    string_columns = [
        "event_time", "event_type", "product_id", "category_id",
        "category_code", "brand", "user_id", "user_session"
    ]

    for column in string_columns:
        blank_string_counts[column] += int(
            chunk[column].str.strip().eq("").fillna(False).sum()
        )

    # Observed event-type values and counts across the complete dataset.
    event_type_counts.update(
        chunk["event_type"].value_counts(dropna=False).to_dict()
    )

    # Date parsing, validity checking, and date coverage.
    parsed_event_time = pd.to_datetime(
        chunk["event_time"],
        errors="coerce",
        utc=True
    )

    invalid_datetime_count += int(
        parsed_event_time.isna().sum() - chunk["event_time"].isna().sum()
    )

    if parsed_event_time.notna().any():
        chunk_earliest = parsed_event_time.min()
        chunk_latest = parsed_event_time.max()

        earliest_event_time = (
            chunk_earliest
            if earliest_event_time is None or chunk_earliest < earliest_event_time
            else earliest_event_time
        )

        latest_event_time = (
            chunk_latest
            if latest_event_time is None or chunk_latest > latest_event_time
            else latest_event_time
        )

    # Price validity checks.
    price_values = chunk["price"]

    price_missing_count += int(price_values.isna().sum())
    price_non_positive_count += int((price_values.le(0)).fillna(False).sum())

    valid_prices = price_values[
        price_values.notna() & price_values.gt(0)
    ]

    if not valid_prices.empty:
        price_valid_count += len(valid_prices)
        price_min = min(price_min, valid_prices.min())
        price_max = max(price_max, valid_prices.max())

    # Reproducible session-level sample for lightweight profiling.
    session_hash = pd.util.hash_pandas_object(
        chunk["user_session"],
        index=False
    )

    sampled_chunk = chunk.loc[
        (session_hash % HASH_MODULUS) == HASH_BUCKET
    ]

    if not sampled_chunk.empty:
        sampled_rows.append(sampled_chunk)

    if chunk_number % 20 == 0:
        print(f"Processed {total_rows:,} rows...")

print(f"\nFull dataset scan completed.")
print(f"Total rows processed: {total_rows:,}")
print(f"Sampled rows retained for profiling: {sum(len(df) for df in sampled_rows):,}")

Processed 5,000,000 rows...
Processed 10,000,000 rows...
Processed 15,000,000 rows...
Processed 20,000,000 rows...
Processed 25,000,000 rows...
Processed 30,000,000 rows...
Processed 35,000,000 rows...
Processed 40,000,000 rows...
Processed 45,000,000 rows...
Processed 50,000,000 rows...
Processed 55,000,000 rows...
Processed 60,000,000 rows...
Processed 65,000,000 rows...

Full dataset scan completed.
Total rows processed: 66,589,268
Sampled rows retained for profiling: 167,146


The full eCommerce dataset was scanned successfully using chunk-based processing to support analysis of a large file without loading all records into memory simultaneously. The dataset contains 66,589,268 event records. A reproducible sample of 167,146 records was retained for lightweight profiling, including exploratory checks of unique values and price distributions.

In [4]:
dataset_overview = pd.DataFrame({
    "metric": [
        "Total event records",
        "Total columns",
        "Earliest valid event timestamp",
        "Latest valid event timestamp",
        "Invalid non-null event timestamps",
        "Valid positive price records",
        "Missing price records",
        "Non-positive price records",
        "Minimum valid price",
        "Maximum valid price"
    ],
    "value": [
        f"{total_rows:,}",
        len(required_columns),
        str(earliest_event_time),
        str(latest_event_time),
        f"{invalid_datetime_count:,}",
        f"{price_valid_count:,}",
        f"{price_missing_count:,}",
        f"{price_non_positive_count:,}",
        f"{price_min:,.2f}" if np.isfinite(price_min) else "Not available",
        f"{price_max:,.2f}" if np.isfinite(price_max) else "Not available"
    ]
})

missing_value_summary = pd.DataFrame({
    "column": required_columns,
    "missing_count": [missing_counts[column] for column in required_columns],
    "blank_string_count": [
        blank_string_counts[column] if column in blank_string_counts else 0
        for column in required_columns
    ]
})

missing_value_summary["missing_percentage"] = (
    missing_value_summary["missing_count"] / total_rows * 100
)

missing_value_summary["blank_string_percentage"] = (
    missing_value_summary["blank_string_count"] / total_rows * 100
)

missing_value_summary = missing_value_summary.sort_values(
    by="missing_count",
    ascending=False
).reset_index(drop=True)

print("Dataset Overview")
display(dataset_overview)

print("Missing-Value Summary")
display(missing_value_summary)

Dataset Overview


,metric,value
0,Total event records,"66,589,268"
1,Total columns,9
2,Earliest valid event timestamp,2020-04-01 00:00:00+00:00
3,Latest valid event timestamp,2020-04-30 23:59:59+00:00
4,Invalid non-null event timestamps,0
5,Valid positive price records,"66,525,177"
6,Missing price records,0
7,Non-positive price records,"64,091"
8,Minimum valid price,0.13
9,Maximum valid price,"2,574.07"


Missing-Value Summary


,column,missing_count,blank_string_count,missing_percentage,blank_string_percentage
0,brand,8992487,0,13.504409,0.0
1,category_code,6755873,0,10.145588,0.0
2,user_session,109,0,0.000164,0.0
3,event_time,0,0,0.000000,0.0
4,event_type,0,0,0.000000,0.0
5,category_id,0,0,0.000000,0.0
6,product_id,0,0,0.000000,0.0
7,price,0,0,0.000000,0.0
8,user_id,0,0,0.000000,0.0


## Dataset Coverage and Initial Data Quality Assessment

The dataset contains 66,589,268 eCommerce event records across nine columns. The available data covers the full period from 1 April 2020, 00:00:00 UTC to 30 April 2020, 23:59:59 UTC.

All event timestamps were successfully parsed, with no invalid non-null timestamp values identified. The `price` variable has no missing values and ranges from 0.13 to 2,574.07 among valid positive records. However, 64,091 records contain non-positive prices. These observations represent approximately 0.10% of the full dataset and will require further inspection before price-based features are created.

Missing values are concentrated in `brand` and `category_code`, with missingness rates of 13.50% and 10.15%, respectively. Since these fields may still provide useful product-level information, missing values will be considered for treatment as an explicit unknown category rather than immediately deleting affected records.

The `user_session` field has only 109 missing values. Since the project uses session-level conversion as the target outcome, these records may need to be excluded during the construction of the modeling dataset because they cannot be reliably assigned to a user session.

In [5]:
event_type_summary = (
    pd.DataFrame.from_dict(
        event_type_counts,
        orient="index",
        columns=["event_count"]
    )
    .rename_axis("event_type")
    .reset_index()
)

event_type_summary["event_percentage"] = (
    event_type_summary["event_count"] / total_rows * 100
)

event_type_summary = event_type_summary.sort_values(
    by="event_count",
    ascending=False
).reset_index(drop=True)

display(event_type_summary)

,event_type,event_count,event_percentage
0,view,62353909,93.639577
1,cart,3268600,4.908599
2,purchase,966759,1.451824


## Observed Customer Event Types

The dataset contains three customer event types: `view`, `cart`, and `purchase`. Product views account for 62,353,909 events or 93.64% of the full dataset. Cart additions account for 3,268,600 events or 4.91%, while purchase events account for 966,759 events or 1.45%.

This distribution shows that the dataset primarily captures browsing activity, with a much smaller proportion of events representing progression toward or completion of a purchase. The `event_type` field will therefore be useful for deriving early-session behavioral features, such as view and cart activity. However, purchase events will be used only to define the session-conversion target and will not be used as model input features, to avoid data leakage.

Although the source description referenced a `remove_from_cart` event type, no such events were observed in the April 2020 dataset.

In [6]:
if not sampled_rows:
    raise ValueError("No rows were retained for the reproducible sample.")

profile_sample = pd.concat(sampled_rows, ignore_index=True)

sample_unique_summary = pd.DataFrame({
    "column": required_columns,
    "sample_non_null_count": [
        profile_sample[column].notna().sum()
        for column in required_columns
    ],
    "sample_unique_count": [
        profile_sample[column].nunique(dropna=True)
        for column in required_columns
    ]
})

valid_sample_prices = profile_sample.loc[
    profile_sample["price"].notna() & profile_sample["price"].gt(0),
    "price"
]

if valid_sample_prices.empty:
    raise ValueError("No valid positive price values were found in the profile sample.")

price_percentiles = valid_sample_prices.quantile(
    [0, 0.01, 0.25, 0.50, 0.75, 0.99, 1.00]
)

q1 = price_percentiles.loc[0.25]
q3 = price_percentiles.loc[0.75]
iqr = q3 - q1
upper_outlier_threshold = q3 + (1.5 * iqr)

sample_high_price_outliers = int(
    valid_sample_prices.gt(upper_outlier_threshold).sum()
)

price_profile_summary = pd.DataFrame({
    "metric": [
        "Valid sampled price records",
        "Minimum price",
        "1st percentile",
        "25th percentile (Q1)",
        "Median price",
        "75th percentile (Q3)",
        "99th percentile",
        "Maximum price",
        "IQR upper outlier threshold",
        "High-price outliers in sample",
        "High-price outlier percentage in sample"
    ],
    "value": [
        f"{len(valid_sample_prices):,}",
        f"{price_percentiles.loc[0.00]:,.2f}",
        f"{price_percentiles.loc[0.01]:,.2f}",
        f"{q1:,.2f}",
        f"{price_percentiles.loc[0.50]:,.2f}",
        f"{q3:,.2f}",
        f"{price_percentiles.loc[0.99]:,.2f}",
        f"{price_percentiles.loc[1.00]:,.2f}",
        f"{upper_outlier_threshold:,.2f}",
        f"{sample_high_price_outliers:,}",
        f"{sample_high_price_outliers / len(valid_sample_prices):.2%}"
    ]
})

print(f"Profile sample size: {len(profile_sample):,} event records")

print("\nSample-Based Unique-Value Summary")
display(sample_unique_summary)

print("\nSample-Based Price Profile and Outlier Assessment")
display(price_profile_summary)

Profile sample size: 167,146 event records

Sample-Based Unique-Value Summary


,column,sample_non_null_count,sample_unique_count
0,event_time,167146,160038
1,event_type,167146,3
2,product_id,167146,36495
3,category_id,167146,913
4,category_code,150165,130
5,brand,145118,2237
6,price,167146,16318
7,user_id,167146,28740
8,user_session,167146,28996



Sample-Based Price Profile and Outlier Assessment


,metric,value
0,Valid sampled price records,"167,017"
1,Minimum price,0.32
2,1st percentile,4.61
3,25th percentile (Q1),51.22
4,Median price,146.31
5,75th percentile (Q3),334.37
6,99th percentile,"1,737.50"
7,Maximum price,"2,574.07"
8,IQR upper outlier threshold,759.10
9,High-price outliers in sample,"15,173"


## Sample-Based Feature Profile and Price Outlier Assessment

A reproducible profile sample containing 167,146 event records was used to assess feature cardinality and the distribution of product prices. The sample contains 28,996 user sessions and 28,740 unique users, confirming that the data is recorded at an event level and that users may generate multiple actions within a session.

The sample includes 36,495 unique products, 913 category IDs, 130 category codes, and 2,237 brands. Product, category, user, and session identifiers are high-cardinality fields and should not be interpreted as continuous numerical variables. They will require aggregation, encoding, or controlled exclusion during later feature engineering.

The product price distribution appears right-skewed. The median sampled price is 146.31, while the 99th percentile is 1,737.50 and the maximum is 2,574.07. Using the interquartile range method, 9.08% of valid sampled price records are classified as high-price outliers. These observations should not be automatically removed because they may represent legitimate premium products. Further preprocessing decisions will focus on validating non-positive prices and considering appropriate transformations or session-level price aggregates.

In [7]:
observed_event_values = ", ".join(
    event_type_summary["event_type"]
    .dropna()
    .astype(str)
    .tolist()
)

data_dictionary_rows = []

for column in required_columns:
    metadata = data_dictionary_metadata[column]

    if column == "event_type":
        observed_values_or_range = observed_event_values
    elif column == "price":
        observed_values_or_range = (
            f"Valid positive values observed from "
            f"{price_min:,.2f} to {price_max:,.2f}"
        )
    elif column == "event_time":
        observed_values_or_range = (
            f"UTC timestamps from {earliest_event_time} to {latest_event_time}"
        )
    elif column in ["product_id", "category_id", "user_id", "user_session"]:
        observed_values_or_range = (
            "High-cardinality identifier; values are not enumerated"
        )
    else:
        observed_values_or_range = (
            "Multiple observed categorical values; refer to sample unique count"
        )

    data_dictionary_rows.append({
        "column": column,
        "description": metadata["description"],
        "semantic_type": metadata["semantic_type"],
        "raw_storage_type": str(profile_sample[column].dtype),
        "format_or_unit": metadata["format_or_unit"],
        "missing_count_full_data": missing_counts[column],
        "missing_percentage_full_data": round(
            missing_counts[column] / total_rows * 100,
            4
        ),
        "sample_unique_count": int(
            profile_sample[column].nunique(dropna=True)
        ),
        "observed_values_or_range": observed_values_or_range,
        "modeling_role": metadata["modeling_role"]
    })

data_dictionary = pd.DataFrame(data_dictionary_rows)

display(data_dictionary)

data_dictionary.to_csv(
    PROCESSED_DATA_FOLDER / "data_dictionary.csv",
    index=False
)

missing_value_summary.to_csv(
    PROCESSED_DATA_FOLDER / "missing_value_summary.csv",
    index=False
)

print(
    "Saved supporting files to:\n"
    f"- {PROCESSED_DATA_FOLDER / 'data_dictionary.csv'}\n"
    f"- {PROCESSED_DATA_FOLDER / 'missing_value_summary.csv'}"
)

,column,description,semantic_type,raw_storage_type,format_or_unit,missing_count_full_data,missing_percentage_full_data,sample_unique_count,observed_values_or_range,modeling_role
0,event_time,UTC timestamp when the customer event occurred.,Datetime / temporal feature,string,UTC date and time,0,0.0000,160038,UTC timestamps from 2020-04-01 00:00:00+00:00 ...,Potential feature for deriving time-based vari...
1,event_type,Type of customer interaction with a product.,Categorical behavioral event,string,Observed event labels,0,0.0000,3,"view, cart, purchase",Potential feature; purchase events will later ...
2,product_id,Unique identifier assigned to a product.,Identifier,string,Product ID,0,0.0000,36495,High-cardinality identifier; values are not en...,Potential feature after aggregation or encoding
3,category_id,Unique identifier assigned to a product category.,Identifier,string,Category ID,0,0.0000,913,High-cardinality identifier; values are not en...,Potential feature after aggregation or encoding
4,category_code,Hierarchical text label describing the product...,Categorical text feature,string,"Dot-separated category hierarchy, when available",6755873,10.1456,130,Multiple observed categorical values; refer to...,Potential feature after cleaning or category-l...
5,brand,"Product brand name, recorded in lowercase when...",Categorical feature,string,Brand name,8992487,13.5044,2237,Multiple observed categorical values; refer to...,Potential feature after missing-value treatmen...
6,price,Recorded price of the product associated with ...,Continuous numeric feature,float64,Price value; currency is not confirmed in the ...,0,0.0000,16318,"Valid positive values observed from 0.13 to 2,...",Potential numeric feature and source for deriv...
7,user_id,Permanent identifier assigned to a user.,Identifier,string,User ID,0,0.0000,28740,High-cardinality identifier; values are not en...,Not intended for direct model use; may support...
8,user_session,Identifier representing one user browsing sess...,Session identifier,string,UUID-like session ID,109,0.0002,28996,High-cardinality identifier; values are not en...,Primary grouping field and future unit of anal...


Saved supporting files to:
- C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\data\processed\data_dictionary.csv
- C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\data\processed\missing_value_summary.csv


# Step 2 Conclusion: Dataset Collection and Understanding

The eCommerce dataset was successfully collected and profiled from the raw file `2020-Apr.csv`. It contains 66,589,268 event-level records and nine variables covering customer activity between 1 April 2020 and 30 April 2020 in UTC.

The dataset captures three observed event types: `view`, `cart`, and `purchase`. Product views account for 93.64% of all events, cart events account for 4.91%, and purchase events account for 1.45%. This confirms that the dataset records the customer journey from browsing through potential purchase completion.

Data-quality checks found no invalid timestamps and no missing price values. However, 64,091 records contain non-positive prices and will require further review during preprocessing. Missingness is concentrated in `brand` at 13.50% and `category_code` at 10.15%. These fields will be retained because they may provide useful predictive information, with missing values represented as an explicit unknown category where appropriate.

The dataset contains high-cardinality identifiers, including product, category, user, and session IDs. These variables will not be treated as continuous numerical values. Instead, they will be used carefully for grouping, aggregation, controlled encoding, or feature derivation.

The price distribution is right-skewed, with a sampled median price of 146.31 and a 99th percentile of 1,737.50. Although 9.08% of valid sampled prices exceed the IQR-based upper outlier threshold, these values may represent genuine premium products and should not be automatically removed.

Overall, the dataset is suitable for the planned session-level purchase-propensity classification project. The next stage will use exploratory data analysis to identify behavioral, temporal, product, and price-related patterns that can guide feature engineering and preprocessing decisions.